In [ ]:
from lago.lago import LinkStream, lago_communities
import pandas as pd

filename = 'syn_data/p0.8_mu0.2_1.csv'

#filename = 'data/CollegeMsg.csv'
df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)
time_links = df.values.tolist()

my_linkstream = LinkStream(is_stream_graph=False)
my_linkstream.add_links(time_links)

print(f"The link stream consists of {my_linkstream.nb_edges} temporal edges (or time links) accross {my_linkstream.nb_nodes} nodes and {my_linkstream.network_duration} time steps, of which only {my_linkstream.nb_timesteps} contain activity.")

The link stream consists of 469 temporal edges (or time links) accross 50 nodes and 2831 time steps, of which only 469 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_77552/601336139.py:7: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(filename, header=None, index_col=False,names=["source","destination","timestamp"], skiprows=1)


In [ ]:
import importlib
from lago.lago import LinkStream, lago_communities

my_linkstream = LinkStream(is_stream_graph=True)
my_linkstream.add_links(time_links)

## Compute dynamic communities
dynamic_communities = lago_communities(
    my_linkstream,
    nb_iter=1,
    )

# Each dynamic community is represented by a list of (<node>, <time instant>)

print(f"{len(dynamic_communities)} dynamic communities have been found")

import importlib
import lago.l_modularity_function as lmf

importlib.reload(lmf)
longitudinal_modularity = lmf.longitudinal_modularity

## Compute Longitudinal Modularity score
## (the higher the better / maximum is 1)
long_mod_score = longitudinal_modularity(
    my_linkstream, 
    dynamic_communities,
    lex_type="MM",
    omega=2
    )

print(f"Longitudinal Modularity score of {long_mod_score} ")

13 dynamic communities have been found
Longitudinal Modularity score of 0.32906 


In [11]:
from __future__ import annotations

from typing import Dict, Hashable, Iterable, Tuple, Any, Optional, Union
import pandas as pd

Node = Hashable
Time = Hashable


def commu_dict_to_interaction_csv(
    commu_dict: Dict[Any, Iterable[Tuple[Node, Time]]],
    syn_csv_path: str = "syn_net.csv",
    out_csv_path: str = "labeled_syn_net.csv",
    *,
    source_col: Union[str, int] = "source",
    destination_col: Union[str, int] = "destination",
    timestamp_col: Union[str, int] = "timestamp",
    syn_header: Optional[int] = "infer",
    syn_sep: str = ",",
    syn_skip_first_row: bool = False,

    node_dtype: Any = int,
    time_dtype: Any = int,

    unknown_commu_value: Any = -1,

    on_conflict: str = "error",
) -> pd.DataFrame:

    syn_df = pd.read_csv(
        syn_csv_path,
        sep=syn_sep,
        header=syn_header,
        skiprows=1 if syn_skip_first_row else None,
        usecols=[source_col, destination_col, timestamp_col],
    ).copy()

    syn_df[source_col] = syn_df[source_col].astype(node_dtype)
    syn_df[destination_col] = syn_df[destination_col].astype(node_dtype)
    syn_df[timestamp_col] = syn_df[timestamp_col].astype(time_dtype)

    active_pairs = set(zip(syn_df[source_col].tolist(), syn_df[timestamp_col].tolist()))
    active_pairs |= set(zip(syn_df[destination_col].tolist(), syn_df[timestamp_col].tolist()))

    node_time_to_commu: Dict[Tuple[Node, Time], Any] = {}

    def _assign(k: Tuple[Node, Time], commu: Any):
        if k not in node_time_to_commu:
            node_time_to_commu[k] = commu
            return
        if node_time_to_commu[k] == commu:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            node_time_to_commu[k] = commu
            return
        raise ValueError(f"Conflict: (node,time)={k} appears in multiple communities: "
                         f"{node_time_to_commu[k]} vs {commu}")

    for commu, pairs in commu_dict.items():
        for u, t in pairs:
            u = node_dtype(u)
            t = time_dtype(t)
            k = (u, t)
            if k in active_pairs:
                _assign(k, commu)

    def _lookup(u: int, t: int):
        return node_time_to_commu.get((u, t), unknown_commu_value)

    syn_df["source_commu"] = [
        _lookup(int(s), int(t)) for s, t in zip(syn_df[source_col], syn_df[timestamp_col])
    ]
    syn_df["destination_commu"] = [
        _lookup(int(d), int(t)) for d, t in zip(syn_df[destination_col], syn_df[timestamp_col])
    ]

    out_df = syn_df.rename(
        columns={
            source_col: "source",
            destination_col: "destination",
            timestamp_col: "timestamp",
        }
    )[["source", "destination", "timestamp", "source_commu", "destination_commu"]]

    out_df.to_csv(out_csv_path, index=False)
    return out_df

real_communities = commu_dict_to_interaction_csv(
    dynamic_communities,
    syn_csv_path="syn_data/p0.8_mu0.2_1.csv",
    out_csv_path="results/p0.8_mu0.2_1/lago.csv",
    syn_header="infer", 
    syn_skip_first_row=False,
    unknown_commu_value=-1, 
    on_conflict="error",
)

(real_communities.head(10))

,source,destination,timestamp,source_commu,destination_commu
0,16,19,5,5,5
1,6,46,11,9,9
2,16,28,18,5,5
3,5,22,25,9,9
4,8,35,31,5,5
5,12,28,37,5,5
6,15,41,43,5,5
7,7,49,52,9,9
8,38,43,60,11,11
9,24,45,63,9,9


In [12]:
from __future__ import annotations

import os
import glob
from typing import Dict, Hashable, Iterable, Tuple, Any, Optional, Union

import pandas as pd
from lago.lago import LinkStream, lago_communities
import lago.l_modularity_function as lmf


Node = Hashable
Time = Hashable


def commu_dict_to_interaction_csv(
    commu_dict: Dict[Any, Iterable[Tuple[Node, Time]]],
    syn_csv_path: str,
    out_csv_path: str,
    *,
    source_col: Union[str, int] = "source",
    destination_col: Union[str, int] = "destination",
    timestamp_col: Union[str, int] = "timestamp",
    syn_header: Optional[int] = "infer",
    syn_sep: str = ",",
    syn_skip_first_row: bool = False,
    node_dtype: Any = int,
    time_dtype: Any = int,
    unknown_commu_value: Any = -1,
    on_conflict: str = "error",
) -> pd.DataFrame:

    syn_df = pd.read_csv(
        syn_csv_path,
        sep=syn_sep,
        header=syn_header,
        skiprows=1 if syn_skip_first_row else None,
        usecols=[source_col, destination_col, timestamp_col],
    ).copy()

    syn_df[source_col] = syn_df[source_col].astype(node_dtype)
    syn_df[destination_col] = syn_df[destination_col].astype(node_dtype)
    syn_df[timestamp_col] = syn_df[timestamp_col].astype(time_dtype)

    active_pairs = set(zip(syn_df[source_col].tolist(), syn_df[timestamp_col].tolist()))
    active_pairs |= set(zip(syn_df[destination_col].tolist(), syn_df[timestamp_col].tolist()))

    node_time_to_commu: Dict[Tuple[Node, Time], Any] = {}

    def _assign(k: Tuple[Node, Time], commu: Any):
        if k not in node_time_to_commu:
            node_time_to_commu[k] = commu
            return
        if node_time_to_commu[k] == commu:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            node_time_to_commu[k] = commu
            return
        raise ValueError(
            f"Conflict: (node,time)={k} appears in multiple communities: "
            f"{node_time_to_commu[k]} vs {commu}"
        )

    for commu, pairs in commu_dict.items():
        for u, t in pairs:
            u = node_dtype(u)
            t = time_dtype(t)
            k = (u, t)
            if k in active_pairs:
                _assign(k, commu)

    def _lookup(u: int, t: int):
        return node_time_to_commu.get((u, t), unknown_commu_value)

    syn_df["source_commu"] = [
        _lookup(int(s), int(t)) for s, t in zip(syn_df[source_col], syn_df[timestamp_col])
    ]
    syn_df["destination_commu"] = [
        _lookup(int(d), int(t)) for d, t in zip(syn_df[destination_col], syn_df[timestamp_col])
    ]

    out_df = syn_df.rename(
        columns={
            source_col: "source",
            destination_col: "destination",
            timestamp_col: "timestamp",
        }
    )[["source", "destination", "timestamp", "source_commu", "destination_commu"]]

    os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
    out_df.to_csv(out_csv_path, index=False)
    return out_df


INPUT_DIR = "syn_data"
RESULTS_DIR = "results"

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))
print(f"Found {len(csv_files)} files in {INPUT_DIR}")

longitudinal_modularity = lmf.longitudinal_modularity

for csv_path in csv_files:
    file_name = os.path.splitext(os.path.basename(csv_path))[0]
    out_dir = os.path.join(RESULTS_DIR, file_name)
    out_csv_path = os.path.join(out_dir, "lago.csv")

    print(f"\nProcessing: {csv_path}")

    df = pd.read_csv(
        csv_path,
        header=None,
        index_col=False,
        names=["source", "destination", "timestamp"],
        skiprows=1,
    )
    time_links = df.values.tolist()

    my_linkstream = LinkStream(is_stream_graph=False)
    my_linkstream.add_links(time_links)

    print(
        f"The link stream consists of {my_linkstream.nb_edges} temporal edges "
        f"(or time links) accross {my_linkstream.nb_nodes} nodes and "
        f"{my_linkstream.network_duration} time steps, of which only "
        f"{my_linkstream.nb_timesteps} contain activity."
    )

    my_linkstream = LinkStream(is_stream_graph=True)
    my_linkstream.add_links(time_links)

    dynamic_communities = lago_communities(
        my_linkstream,
        nb_iter=1,
    )

    print(f"{len(dynamic_communities)} dynamic communities have been found")

    long_mod_score = longitudinal_modularity(
        my_linkstream,
        dynamic_communities,
        lex_type="MM",
        omega=2,
    )

    print(f"Longitudinal Modularity score of {long_mod_score}")

    real_communities = commu_dict_to_interaction_csv(
        dynamic_communities,
        syn_csv_path=csv_path,
        out_csv_path=out_csv_path,
        syn_header="infer",
        syn_skip_first_row=False,
        unknown_commu_value=-1,
        on_conflict="error",
    )

    print(f"Saved: {out_csv_path}")
    print(real_communities.head())

Found 100 files in syn_data

Processing: syn_data/p0.85_mu0.05_1.csv
The link stream consists of 473 temporal edges (or time links) accross 50 nodes and 2849 time steps, of which only 473 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_77552/4171142549.py:112: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


8 dynamic communities have been found
Longitudinal Modularity score of 0.46965
Saved: results/p0.85_mu0.05_1/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0      15           20          9             0                  0
1       4           25         15             7                  7
2      24           32         21             0                  0
3       8           10         28             7                  7
4      16           49         35             7                  7

Processing: syn_data/p0.85_mu0.05_2.csv
The link stream consists of 479 temporal edges (or time links) accross 50 nodes and 2877 time steps, of which only 479 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_77552/4171142549.py:112: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


8 dynamic communities have been found
Longitudinal Modularity score of 0.5156
Saved: results/p0.85_mu0.05_2/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0       5           11          6             6                  6
1       4           11         13             6                  6
2      17           35         20             4                  4
3       0           32         26             3                  3
4      17           28         32             4                  4

Processing: syn_data/p0.85_mu0.05_3.csv
The link stream consists of 450 temporal edges (or time links) accross 50 nodes and 2711 time steps, of which only 450 contain activity.


...

4 dynamic communities have been found
Longitudinal Modularity score of 0.59544
Saved: results/p1.0_mu0.2_2/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0       7           24          4             1                  1
1       8           13         13             3                  3
2       6           46         18             2                  2
3      12           39         29             2                  2
4       0           22         35             0                  0

Processing: syn_data/p1.0_mu0.2_3.csv
The link stream consists of 510 temporal edges (or time links) accross 50 nodes and 3045 time steps, of which only 510 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_77552/4171142549.py:112: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


6 dynamic communities have been found
Longitudinal Modularity score of 0.51465
Saved: results/p1.0_mu0.2_3/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0      16           29          7             5                  5
1       5           32         13             2                  2
2      30           47         19             0                  0
3       4           14         28             0                  0
4       7           18         36             1                  1

Processing: syn_data/p1.0_mu0.2_4.csv
The link stream consists of 457 temporal edges (or time links) accross 50 nodes and 2763 time steps, of which only 457 contain activity.


/var/folders/g1/wg9xf7ks75b49ljdm6l5zy280000gp/T/ipykernel_77552/4171142549.py:112: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


8 dynamic communities have been found
Longitudinal Modularity score of 0.4058
Saved: results/p1.0_mu0.2_4/lago.csv
   source  destination  timestamp  source_commu  destination_commu
0      18           22          6             0                  0
1       6           32         17             7                  7
2      18           37         24             0                  0
3       6           17         30             7                  7
4      10           11         37             0                  0
